<a href="https://colab.research.google.com/github/swathipraveen/Python_practice/blob/Python/RecommenderSystemsAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time
from sklearn.model_selection import train_test_split
from scipy.sparse import coo_matrix, csr_matrix
from scipy.spatial.distance import jaccard, cosine
from pytest import approx
ratings = np.array([[3, 3, 4, 0, 4, 2, 3, 0],
                    [3, 5, 4, 3, 3, 0, 0, 4],
                    [0, 4, 0, 5, 0, 0, 2, 1],
                    [2, 0, 0, 4, 0, 4, 4, 5]])
Ub = (ratings>0).astype(int)
print(Ub)

def jaccard(a,b):
    return (a*b).sum()/((a+b)>0).sum()

users = ["firechicken","mike0702","zephyros","dadvador"]

simmat=np.zeros((4,4))
for i in range(4):
    for j in range(4):
        simmat[i,j] = jaccard(Ub[i],Ub[j])
        if i<j:
            print(users[i]+'-'+users[j], jaccard(Ub[i],Ub[j]))

print(simmat)

[[1 1 1 0 1 1 1 0]
 [1 1 1 1 1 0 0 1]
 [0 1 0 1 0 0 1 1]
 [1 0 0 1 0 1 1 1]]
firechicken-mike0702 0.5
firechicken-zephyros 0.25
firechicken-dadvador 0.375
mike0702-zephyros 0.42857142857142855
mike0702-dadvador 0.375
zephyros-dadvador 0.5
[[1.         0.5        0.25       0.375     ]
 [0.5        1.         0.42857143 0.375     ]
 [0.25       0.42857143 1.         0.5       ]
 [0.375      0.375      0.5        1.        ]]


In [3]:
def cos(a,b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

simmat=np.zeros((4,4))
for i in range(4):
    for j in range(4):
        simmat[i,j] = cos(Ub[i],Ub[j])
        if i<j:
            print(users[i]+'-'+users[j], cos(Ub[i],Ub[j]))
print(simmat)

firechicken-mike0702 0.6666666666666667
firechicken-zephyros 0.4082482904638631
firechicken-dadvador 0.5477225575051661
mike0702-zephyros 0.6123724356957946
mike0702-dadvador 0.5477225575051661
zephyros-dadvador 0.6708203932499369
[[1.         0.66666667 0.40824829 0.54772256]
 [0.66666667 1.         0.61237244 0.54772256]
 [0.40824829 0.61237244 1.         0.67082039]
 [0.54772256 0.54772256 0.67082039 1.        ]]


In [4]:
Ur = (ratings>=3).astype(int)
print(Ur)

simmat=np.zeros((4,4))
for i in range(4):
    for j in range(4):
        simmat[i,j] = jaccard(Ur[i],Ur[j])
        if i<j:
            print(users[i]+'-'+users[j], jaccard(Ur[i],Ur[j]))

[[1 1 1 0 1 0 1 0]
 [1 1 1 1 1 0 0 1]
 [0 1 0 1 0 0 0 0]
 [0 0 0 1 0 1 1 1]]
firechicken-mike0702 0.5714285714285714
firechicken-zephyros 0.16666666666666666
firechicken-dadvador 0.125
mike0702-zephyros 0.3333333333333333
mike0702-dadvador 0.25
zephyros-dadvador 0.2


In [5]:
class RecSys():
    def __init__(self,data):
        self.data=data
        self.allusers = list(self.data.users['uID'])
        self.allmovies = list(self.data.movies['mID'])
        self.genres = list(self.data.movies.columns.drop(['mID', 'title', 'year']))
        self.mid2idx = dict(zip(self.data.movies.mID,list(range(len(self.data.movies)))))
        self.uid2idx = dict(zip(self.data.users.uID,list(range(len(self.data.users)))))
        self.Mr=self.rating_matrix()
        self.Mm=None
        self.sim=np.zeros((len(self.allmovies),len(self.allmovies)))

    def rating_matrix(self):
        """
        Convert the rating matrix to numpy array of shape (#allusers,#allmovies)
        """
        ind_movie = [self.mid2idx[x] for x in self.data.train.mID]
        ind_user = [self.uid2idx[x] for x in self.data.train.uID]
        rating_train = list(self.data.train.rating)

        return np.array(coo_matrix((rating_train, (ind_user, ind_movie)), shape=(len(self.allusers), len(self.allmovies))).toarray())


    def predict_everything_to_3(self):
        """
        Predict everything to 3 for the test data
        """
        # Generate an array with 3s against all entries in test dataset
        # your code here
        return np.full(len(self.data.test), 3.0)

    def predict_to_user_average(self):
        """
        Predict to average rating for the user.
        Returns numpy array of shape (#users,)
        """
        # Generate an array as follows:
        # 1. Calculate all avg user rating as sum of ratings of user across all movies/number of movies whose rating > 0
        # 2. Return the average rating of users in test data
        # your code here
        user_sums = self.Mr.sum(axis=1)
        user_counts = (self.Mr > 0).sum(axis=1)
        user_avg = np.where(user_counts > 0, user_sums / user_counts, 0)
        test_user_indices = [self.uid2idx[u] for u in self.data.test.uID]
        return user_avg[test_user_indices]
        pass

    def predict_from_sim(self,uid,mid):
        """
        Predict a user rating on a movie given userID and movieID
        """
        # Predict user rating as follows:
        # 1. Get entry of user id in rating matrix
        # 2. Get entry of movie id in sim matrix
        # 3. Employ 1 and 2 to predict user rating of the movie
        # your code here
        u = self.uid2idx[uid]
        m = self.mid2idx[mid]
        user_ratings = self.Mr[u, :]
        movie_sims = self.sim[m, :]
        rated_mask = user_ratings > 0

        if rated_mask.sum() == 0:
            return 3.0

        rated_ratings = user_ratings[rated_mask]
        rated_sims = movie_sims[rated_mask]

        if rated_sims.sum() == 0:
            return rated_ratings.mean()

        prediction = np.dot(rated_sims, rated_ratings) / rated_sims.sum()

        return prediction
        pass

    def predict(self):
        """
        Predict ratings in the test data. Returns predicted rating in a numpy array of size (# of rows in testdata,)
        """
        # your code here
        preds = []

        for _, row in self.data.test.iterrows():
            uid = row.uID
            mid = row.mID
            pred = self.predict_from_sim(uid, mid)
            preds.append(pred)

        return np.array(preds)

        pass

    def rmse(self,yp):
        yp[np.isnan(yp)]=3 #In case there is nan values in prediction, it will impute to 3.
        yt=np.array(self.data.test.rating)
        return np.sqrt(((yt-yp)**2).mean())


class ContentBased(RecSys):
    def __init__(self,data):
        super().__init__(data)
        self.data=data
        self.Mm = self.calc_movie_feature_matrix()

    def calc_movie_feature_matrix(self):
        """
        Create movie feature matrix in a numpy array of shape (#allmovies, #genres)
        """
        # your code here
        movie_features = self.data.movies[self.genres].values

        return movie_features.astype(float)

        pass

    def calc_item_item_similarity(self):
        """
        Create item-item similarity using Jaccard similarity
        """
        # Update the sim matrix by calculating item-item similarity using Jaccard similarity
        # Jaccard Similarity: J(A, B) = |A∩B| / |A∪B|
        # your code here
        M = self.Mm
        n_movies = M.shape[0]

        sim = np.zeros((n_movies, n_movies))

        for i in range(n_movies):
            for j in range(i, n_movies):  # compute only upper triangle
                A = M[i]
                B = M[j]

                intersection = np.sum((A == 1) & (B == 1))
                union = np.sum((A == 1) | (B == 1))

                if union == 0:
                    s = 0.0
                else:
                    s = intersection / union

                sim[i, j] = sim[j, i] = s  # symmetric

        self.sim = sim
        return sim


class Collaborative(RecSys):
    def __init__(self,data):
        super().__init__(data)

    def calc_item_item_similarity(self, simfunction, *X):
        """
        Create item-item similarity using similarity function.
        X is an optional transformed matrix of Mr
        """
        # General function that calculates item-item similarity based on the sim function and data inputed
        if len(X)==0:
            self.sim = simfunction()
        else:
            self.sim = simfunction(X[0]) # *X passes in a tuple format of (X,), to X[0] will be the actual transformed matrix

    def cossim(self):
        """
        Calculates item-item similarity for all pairs of items using cosine similarity (values from 0 to 1) on utility matrix
        Returns a cosine similarity matrix of size (#all movies, #all movies)
        """
        # Return a sim matrix by calculating item-item similarity for all pairs of items using Jaccard similarity
        # Cosine Similarity: C(A, B) = (A.B) / (||A||.||B||)
        # your code here
        M = self.Mr.T
        dot_products = M @ M.T

        norms = np.linalg.norm(M, axis=1)
        norm_matrix = np.outer(norms, norms)


        sim = np.divide(
            dot_products,
            norm_matrix,
            out=np.zeros_like(dot_products, dtype=float),
            where=norm_matrix != 0
        )

        return sim

        pass

    def jacsim(self,Xr):
        """
        Calculates item-item similarity for all pairs of items using jaccard similarity (values from 0 to 1)
        Xr is the transformed rating matrix.
        """
        # Return a sim matrix by calculating item-item similarity for all pairs of items using Jaccard similarity
        # Jaccard Similarity: J(A, B) = |A∩B| / |A∪B|
        # your code here
        M = Xr.T.astype(bool)

        intersection = M @ M.T

        movie_counts = M.sum(axis=1)  # (#movies,)
        union = (
            movie_counts.reshape(-1, 1) +
            movie_counts.reshape(1, -1) -
            intersection
        )

        sim = np.divide(
            intersection,
            union,
            out=np.zeros_like(intersection, dtype=float),
            where=union != 0
        )

        return sim

        pass

